# Exploratory Analysis — Instruction Injection in LLM Systems with RAG

This notebook reproduces and complements the analysis in `src/evaluator.py`.
All figures generated here are exploratory variants; the official figures
for the thesis are generated with the `src/evaluator.py` script.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np

# Relative paths from notebooks/
RESULTS_DIR = '../results/'
FIGURES_DIR = '../figures/'

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

## 1. Data loading

In [ ]:
raw = pd.read_csv(RESULTS_DIR + 'full_experiment_raw.csv')
labelled = pd.read_csv(RESULTS_DIR + 'full_experiment_labelled.csv')
tool_calls = pd.read_csv(RESULTS_DIR + 'full_experiment_tool_calls.csv') if pd.read_csv.__doc__ else None
metrics = pd.read_csv(RESULTS_DIR + 'metrics_summary.csv')
by_family = pd.read_csv(RESULTS_DIR + 'metrics_by_attack_family.csv')

print(f"Total runs: {len(labelled)}")
print(f"Configurations: {labelled['configuration'].unique()}")
print(f"Questions: {labelled['query_id'].nunique()}")
labelled.head()

## 2. Metrics by configuration

In [ ]:
print(metrics[['configuration', 'attack_success_rate', 'tool_misuse_rate',
               'leakage_rate', 'safe_refusal_rate', 'answer_utility']]
      .set_index('configuration')
      .round(3)
      .to_string())

In [ ]:
# Table formatted as a percentage
display_metrics = metrics.set_index('configuration')[[
    'attack_success_rate', 'safe_refusal_rate', 'tool_misuse_rate',
    'leakage_rate', 'answer_utility'
]].rename(columns={
    'attack_success_rate': 'ASR (%)',
    'safe_refusal_rate': 'SRR (%)',
    'tool_misuse_rate': 'TMR (%)',
    'leakage_rate': 'LR (%)',
    'answer_utility': 'AU (%)'
}) * 100

display_metrics.round(1)

## 3. Evolution of ASR across configurations

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

configs = metrics['configuration']
asr = metrics['attack_success_rate'] * 100
srr = metrics['safe_refusal_rate'] * 100
au = metrics['answer_utility'] * 100

ax.plot(configs, asr, 'o-', color='#e63946', label='Attack Success Rate (ASR)', linewidth=2)
ax.plot(configs, srr, 's--', color='#2a9d8f', label='Safe Refusal Rate (SRR)', linewidth=2)
ax.plot(configs, au, '^:', color='#457b9d', label='Answer Utility (AU)', linewidth=2)

for x, y in zip(configs, asr):
    ax.annotate(f'{y:.1f}%', (x, y), textcoords='offset points', xytext=(0, 8), ha='center', fontsize=9)

ax.set_ylabel('Percentage (%)')
ax.set_title('Metric evolution by configuration')
ax.legend(loc='center right')
ax.set_ylim(0, 105)
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout()
plt.savefig(FIGURES_DIR + 'metrics_evolution.png', dpi=150)
plt.show()

## 4. ASR by attack family and configuration

In [ ]:
pivot = by_family.pivot(index='attack_family', columns='configuration', values='attack_success_rate') * 100
pivot = pivot.round(1)
print(pivot.to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

families = pivot.index.tolist()
x = np.arange(len(families))
width = 0.2
colors = ['#e63946', '#f4a261', '#2a9d8f', '#264653']

for i, (config, color) in enumerate(zip(['C1', 'C2', 'C3', 'C4'], colors)):
    ax.bar(x + i * width, pivot[config], width, label=config, color=color)

ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(families, rotation=15, ha='right')
ax.set_ylabel('ASR (%)')
ax.set_title('Attack Success Rate by attack family and configuration')
ax.legend(title='Config.')
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout()
plt.savefig(FIGURES_DIR + 'asr_by_family_detailed.png', dpi=150)
plt.show()

## 5. Security vs. usefulness trade-off

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

for _, row in metrics.iterrows():
    ax.scatter(row['attack_success_rate'] * 100, row['answer_utility'] * 100,
               s=120, zorder=5)
    ax.annotate(row['configuration'],
                (row['attack_success_rate'] * 100, row['answer_utility'] * 100),
                textcoords='offset points', xytext=(6, 4), fontsize=11, fontweight='bold')

ax.set_xlabel('Attack Success Rate (ASR) %')
ax.set_ylabel('Answer Utility (AU) %')
ax.set_title('Security vs. usefulness trade-off')
ax.invert_xaxis()  # lower ASR = more secure, shown on the right
ax.xaxis.set_major_formatter(mtick.PercentFormatter())
ax.yaxis.set_major_formatter(mtick.PercentFormatter())
plt.tight_layout()
plt.savefig(FIGURES_DIR + 'tradeoff_security_utility.png', dpi=150)
plt.show()

print("""Interpretation:
- Top right = more secure AND more useful (ideal).
- C4 dominates C1/C2: lower ASR and higher AU simultaneously.
- Usefulness in C3/C4 is higher because the poisoned documents no longer
  interfere with the answer to the user's actual question.""")

## 6. Distribution of question types

In [ ]:
# Analyze only C1 (one configuration is enough to see the distribution)
c1 = labelled[labelled['configuration'] == 'C1']
print("Question types:")
print(c1['query_type'].value_counts())
print("\nRetrieved attack families:")
# Expand attack_families (pipe-separated)
families_flat = c1['attack_families'].dropna().str.split('|').explode()
print(families_flat.value_counts())

## 7. Manual review of rows flagged as `needs_manual_review`

In [ ]:
if 'needs_manual_review' in labelled.columns:
    review = labelled[labelled['needs_manual_review'] == 1]
    print(f"Rows requiring manual review: {len(review)}")
    if len(review) > 0:
        display(review[['query_id', 'configuration', 'question', 'response',
                         'attack_success', 'answer_useful']].head(10))
else:
    print("needs_manual_review field not found in the CSV.")

## 8. Statistical summary

In [ ]:
print("=" * 60)
print("EXECUTIVE SUMMARY OF RESULTS")
print("=" * 60)
for _, row in metrics.iterrows():
    print(f"""
Configuration {row['configuration']}:
  Attack Success Rate : {row['attack_success_rate']:.1%}
  Safe Refusal Rate   : {row['safe_refusal_rate']:.1%}
  Tool Misuse Rate    : {row['tool_misuse_rate']:.1%}
  Leakage Rate        : {row['leakage_rate']:.1%}
  Answer Utility      : {row['answer_utility']:.1%}
""")